# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Build the feature vector with categorical handling and missing-value fills
feat = con.sql(f"""
    SELECT content_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           AVG(ga4_engaged_sessions) as avg_engaged_sessions,
           SUM(ga4_pageviews) as total_pageviews,
           COUNT(*) as days_seen,
           MAX(client_has_gsc) as client_has_gsc
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

# Handle missing values explicitly (fill numeric with 0, since missing = no activity that day)
feat = feat.fillna(0)

# Categorical handling: client_has_gsc is boolean -- convert to int for modeling
feat["client_has_gsc"] = feat["client_has_gsc"].astype(int)

print(f"Feature vector built: {len(feat):,} rows, {feat.shape[1]} columns")
print(feat.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector built: 176,738 rows, 8 columns
            content_hash_id  avg_position  total_impressions  total_clicks  \
0  content_7a105f548d9c6916      7.209549             6523.0           7.0   
1  content_a3ea9792f793ec72      2.987198              453.0           0.0   
2  content_36c36abc7650d7af      6.724039             5630.0           6.0   
3  content_a7da352b73b02668      7.244844             4944.0          13.0   
4  content_1855a661b4d36130      4.209227              429.0           1.0   

   avg_engaged_sessions  total_pageviews  days_seen  client_has_gsc  
0                   0.0              1.0         31               1  
1                   0.0              0.0         31               1  
2                   0.0              6.0         31               1  
3                   0.0              2.0         31               1  
4                   0.0              2.0         31               1  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature notes:")
print()
print("avg_position (float): average search ranking position over March.")
print("  Missing handling: filled with 0 if no GSC data that day (rare, since")
print("  we already filtered to gsc_data_available IS TRUE).")
print("  Available: at the end of each day, once GSC reports land -- known before")
print("  any 'is this page declining' decision is made.")
print()
print("total_impressions / total_clicks (float, summed): raw traffic volume.")
print("  Missing handling: SUM naturally treats missing days as 0 contribution.")
print("  Available: accumulated daily, always known before the decision moment.")
print()
print("avg_engaged_sessions / total_pageviews (float): GA4 engagement signals.")
print("  Missing handling: filled with 0 -- absence means no GA4 activity, not")
print("  unknown data (client_has_gsc still true, but GA4 activity can be zero).")
print("  Available: same-day GA4 data, known before the decision.")
print()
print("days_seen (int): count of days with data in the window -- no missing")
print("  values possible since it's a COUNT.")
print()
print("client_has_gsc (categorical/boolean -> int): whether the client has GSC")
print("  connected at all. Converted to 0/1 for modeling. Available at all times")
print("  since it's a client-level property, not day-specific.")

Feature notes:

avg_position (float): average search ranking position over March.
  Missing handling: filled with 0 if no GSC data that day (rare, since
  we already filtered to gsc_data_available IS TRUE).
  Available: at the end of each day, once GSC reports land -- known before
  any 'is this page declining' decision is made.

total_impressions / total_clicks (float, summed): raw traffic volume.
  Missing handling: SUM naturally treats missing days as 0 contribution.
  Available: accumulated daily, always known before the decision moment.

avg_engaged_sessions / total_pageviews (float): GA4 engagement signals.
  Missing handling: filled with 0 -- absence means no GA4 activity, not
  unknown data (client_has_gsc still true, but GA4 activity can be zero).
  Available: same-day GA4 data, known before the decision.

days_seen (int): count of days with data in the window -- no missing
  values possible since it's a COUNT.

client_has_gsc (categorical/boolean -> int): whether the client h

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

# Build the label: bottom 30% by total_clicks = "declining"
feat["is_declining"] = (feat["total_clicks"] <= feat["total_clicks"].quantile(0.30)).astype(int)

honest_cols = ["avg_position", "total_impressions", "avg_engaged_sessions",
               "total_pageviews", "days_seen", "client_has_gsc"]
X_honest = feat[honest_cols]
y = feat["is_declining"]

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_honest, y)
honest_score = tree.score(X_honest, y)
print(f"HONEST baseline accuracy: {honest_score:.3f}")
print()

# --- ATTACK 1: label-derived column ---
X_leak1 = feat[honest_cols + ["total_clicks"]]
tree1 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leak1, y)
score1 = tree1.score(X_leak1, y)
print(f"ATTACK 1 -- label-derived column (total_clicks): {score1:.3f}")
print("  Why it's a leak: total_clicks literally defines is_declining. Including")
print("  it lets the model read the label almost directly.")
print()

# --- ATTACK 2: future window ---
# Simulate an April (future) signal that wouldn't be known in March
future = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_clicks) as future_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_hash_id
""").df()
feat_future = feat.merge(future, on="content_hash_id", how="left").fillna(0)
X_leak2 = feat_future[honest_cols + ["future_clicks"]]
tree2 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leak2, y)
score2 = tree2.score(X_leak2, y)
print(f"HONEST score kept after removing both leaks: {honest_score:.3f}")
print()
print("Note: Attack 1 (label-derived) was a dramatic leak (0.842 -> 1.000) --")
print("unmistakable overfitting to the label definition itself.")
print("Attack 2 (future window) only moved the score slightly (0.842 -> 0.850) --")
print("a weaker signal here, but still conceptually wrong: future data should")
print("never be used, regardless of how much it helps the score.")

HONEST baseline accuracy: 0.842

ATTACK 1 -- label-derived column (total_clicks): 1.000
  Why it's a leak: total_clicks literally defines is_declining. Including
  it lets the model read the label almost directly.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST score kept after removing both leaks: 0.842

Note: Attack 1 (label-derived) was a dramatic leak (0.842 -> 1.000) --
unmistakable overfitting to the label definition itself.
Attack 2 (future window) only moved the score slightly (0.842 -> 0.850) --
a weaker signal here, but still conceptually wrong: future data should
never be used, regardless of how much it helps the score.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Excluded fields and why:")
print()
print("trend_direction / trend_pct (if present) -- pre-computed outcome signals")
print("  that likely encode the label itself; using them would repeat Attack 1's")
print("  leak in a hidden form.")
print()
print("client_hash_id -- a raw identifier, not a generalizable feature; including")
print("  it risks the model memorizing specific clients instead of learning")
print("  transferable patterns.")
print()
print("ai_chatgpt / ai_perplexity / etc (AI referral breakdowns) -- too sparse and")
print("  noisy at this early stage; most pages have near-zero AI traffic in March")
print("  2026, so these columns add little signal but real overfitting risk.")
print()
print("Any April+ (future) data -- excluded on principle after Attack 2, even")
print("  though the leak was mild here, using future data breaks the past-to-")
print("  future prediction setup this whole task depends on.")

Excluded fields and why:

trend_direction / trend_pct (if present) -- pre-computed outcome signals
  that likely encode the label itself; using them would repeat Attack 1's
  leak in a hidden form.

client_hash_id -- a raw identifier, not a generalizable feature; including
  it risks the model memorizing specific clients instead of learning
  transferable patterns.

ai_chatgpt / ai_perplexity / etc (AI referral breakdowns) -- too sparse and
  noisy at this early stage; most pages have near-zero AI traffic in March
  2026, so these columns add little signal but real overfitting risk.

Any April+ (future) data -- excluded on principle after Attack 2, even
  though the leak was mild here, using future data breaks the past-to-
  future prediction setup this whole task depends on.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.